# 03 — GGUF Conversion and 4-Model Evaluation

Converts SFT and DPO models to GGUF Q4_K_M, then benchmarks four models on the held-out test split.

**Prerequisites:** Notebooks 01 and 02 must be complete. `test_split.json` must exist (created by Notebook 00).

In [ ]:
# ─── CONFIG — edit this cell only ────────────────────────
import os, sys
from pathlib import Path

MODEL_SIZE = '7b'
DRIVE_PATH = '/content/drive/MyDrive/smartriz/'
REPO_PATH  = '/content/smartriz-project'
if Path(REPO_PATH).exists() and REPO_PATH + '/src' not in sys.path:
    sys.path.insert(0, REPO_PATH + '/src')

try:
    from google.colab import userdata
    DEEPINFRA_API_KEY = userdata.get('DEEPINFRA_API_KEY') or os.getenv('DEEPINFRA_API_KEY', '')
except Exception:
    DEEPINFRA_API_KEY = os.getenv('DEEPINFRA_API_KEY', '')

if not DEEPINFRA_API_KEY:
    print('WARNING: DEEPINFRA_API_KEY not set — teacher eval and LLM judge will be skipped.')

SFT_MODEL_DIR  = f'{DRIVE_PATH}checkpoints/sft-{MODEL_SIZE}/merged/'
DPO_MODEL_DIR  = f'{DRIVE_PATH}checkpoints/dpo-{MODEL_SIZE}/merged/'
GGUF_DIR       = f'{DRIVE_PATH}gguf/'
EVAL_DIR       = f'{DRIVE_PATH}evaluation/'
TEST_PATH      = f'{DRIVE_PATH}data/test_split.json'
BASELINE_MODEL = f'Qwen/Qwen2.5-{MODEL_SIZE.upper()}-Instruct'
TEACHER_MODEL  = 'deepseek-ai/DeepSeek-R1-Distill-Llama-70B'

RUN_LLM_JUDGE = True
JUDGE_MODEL   = 'deepseek-ai/DeepSeek-R1-Distill-Llama-70B'

# Regression gate thresholds — DPO must clear these before GGUF conversion
GATE_FORMAT_MIN     = 0.85
GATE_PREAMBLE_MAX   = 0.10
GATE_SAMPLE_COUNT   = 5


In [ ]:
!pip install -q transformers accelerate sentence-transformers scikit-learn openai tqdm

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
os.makedirs(GGUF_DIR, exist_ok=True)
os.makedirs(EVAL_DIR, exist_ok=True)

In [ ]:
# ── Clone repo so smartriz helpers (format_check, formatter) are importable ──
# Required because Cell 1 set REPO_PATH but the actual code lives in your
# GitHub repo, not in Drive. Adjust REPO_URL below to match your fork.
#   - Public repo:  https://github.com/<user>/smartriz-project.git
#   - Private repo: https://<TOKEN>@github.com/<user>/smartriz-project.git
import sys
from pathlib import Path

REPO_URL = 'https://github.com/sevketugurel/smartriz-project.git'

if not Path(REPO_PATH).exists():
    print(f'Cloning {REPO_URL} → {REPO_PATH}')
    !git clone --depth 1 {REPO_URL} {REPO_PATH}
else:
    print(f'Repo already present at {REPO_PATH} — pulling latest')
    !cd {REPO_PATH} && git pull --ff-only

if REPO_PATH + '/src' not in sys.path:
    sys.path.insert(0, REPO_PATH + '/src')

from smartriz.training.formatter import SYSTEM_PROMPT  # noqa: F401
from smartriz.eval.format_check import score_format_compliance  # noqa: F401
print('Repo helpers import OK')

In [ ]:
# ── DPO regression gate (runs ONLY if DPO merged exists) ──────────────────
# Quick 5-sample inference; aborts if format compliance or chat-preamble
# leak rate exceeds the configured thresholds. This protects downstream
# GGUF conversion from publishing a regressed adapter.
import json, gc, torch
from pathlib import Path
from smartriz.eval.format_check import score_format_compliance, aggregate_scores
from smartriz.training.formatter import SYSTEM_PROMPT

if not Path(DPO_MODEL_DIR + 'config.json').exists():
    print(f'No DPO merged model at {DPO_MODEL_DIR} — skipping gate.')
else:
    from transformers import pipeline as hf_pipeline, AutoTokenizer
    with open(TEST_PATH) as f:
        gate_cases = json.load(f)['cases'][:GATE_SAMPLE_COUNT]
    tok = AutoTokenizer.from_pretrained(DPO_MODEL_DIR, trust_remote_code=True)
    gen = hf_pipeline('text-generation', model=DPO_MODEL_DIR, tokenizer=tok,
                      device_map='auto', max_new_tokens=1024, do_sample=False)
    try:
        outputs = []
        for c in gate_cases:
            prompt = tok.apply_chat_template(
                [{'role': 'system', 'content': SYSTEM_PROMPT},
                 {'role': 'user',   'content': c['problem']}],
                tokenize=False, add_generation_prompt=True
            )
            text = gen(prompt)[0]['generated_text']
            outputs.append(text[len(prompt):].strip())
    finally:
        del gen
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    scores = [score_format_compliance(o) for o in outputs]
    agg = aggregate_scores(scores)
    print('DPO regression gate:')
    print(json.dumps(agg, indent=2))

    if agg['mean'] < GATE_FORMAT_MIN:
        raise AssertionError(
            f'DPO format_compliance.mean={agg["mean"]:.3f} < '
            f'{GATE_FORMAT_MIN} — abort GGUF conversion. Review w&b reward '
            'curves; consider beta=0.05 or loss_type="sigmoid" rerun.'
        )
    if agg['chat_preamble_leak_rate'] > GATE_PREAMBLE_MAX:
        raise AssertionError(
            f'DPO chat_preamble_leak_rate={agg["chat_preamble_leak_rate"]:.3f} > '
            f'{GATE_PREAMBLE_MAX} — abort. Increase stream A:chat_preamble share '
            'in build_dpo_pairs.py and rerun DPO.'
        )
    print(f'Gate PASS — proceeding to GGUF conversion.')


In [ ]:
# ── GGUF conversion with imatrix calibration ──────────────────────────────
# imatrix guides Q4_K_M quantization with importance data from real examples,
# improving perplexity by ~2-5% vs direct conversion — matters for reasoning tasks.
import json
import torch
from pathlib import Path

SFT_GGUF_F16 = f'{GGUF_DIR}smartriz-sft-{MODEL_SIZE}-F16.gguf'
DPO_GGUF_F16 = f'{GGUF_DIR}smartriz-dpo-{MODEL_SIZE}-F16.gguf'
SFT_GGUF     = f'{GGUF_DIR}smartriz-sft-{MODEL_SIZE}-Q4_K_M.gguf'
DPO_GGUF     = f'{GGUF_DIR}smartriz-dpo-{MODEL_SIZE}-Q4_K_M.gguf'
SFT_IMATRIX  = f'{GGUF_DIR}sft-imatrix.dat'
DPO_IMATRIX  = f'{GGUF_DIR}dpo-imatrix.dat'
CALIB_FILE   = f'{GGUF_DIR}calibration.txt'

GPU_LAYERS = 99 if torch.cuda.is_available() else 0

# ── Clone and build llama.cpp ─────────────────────────────────────────────
if not Path('/content/llama.cpp').exists():
    print('Cloning and building llama.cpp...')
    !git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp
    !cmake /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=ON -DCMAKE_BUILD_TYPE=Release
    !cmake --build /content/llama.cpp/build --config Release -j $(nproc)
    !pip install -q /content/llama.cpp

LLAMA_IMATRIX = '/content/llama.cpp/build/bin/llama-imatrix'
LLAMA_QUANT   = '/content/llama.cpp/build/bin/llama-quantize'

# ── Generate calibration data from test split ─────────────────────────────
if not Path(CALIB_FILE).exists():
    with open(TEST_PATH) as f:
        calib_cases = json.load(f)['cases'][:100]
    lines = [f'Problem: {c["problem"]}\nSolution: {c["solution"]}\n' for c in calib_cases]
    with open(CALIB_FILE, 'w') as f:
        f.write('\n'.join(lines))
    print(f'Calibration data written: {len(calib_cases)} examples')

# ── SFT model: F16 → imatrix → Q4_K_M ───────────────────────────────────
if not Path(SFT_GGUF).exists():
    if not Path(SFT_GGUF_F16).exists():
        print('Converting SFT → GGUF F16...')
        !python /content/llama.cpp/convert_hf_to_gguf.py \
            {SFT_MODEL_DIR} --outfile {SFT_GGUF_F16} --outtype f16
    if not Path(SFT_IMATRIX).exists():
        print('Generating SFT imatrix (~15-20 min)...')
        !{LLAMA_IMATRIX} -m {SFT_GGUF_F16} -f {CALIB_FILE} \
            -o {SFT_IMATRIX} --n-gpu-layers {GPU_LAYERS} --chunks 128
    print('Quantizing SFT → Q4_K_M with imatrix...')
    !{LLAMA_QUANT} --imatrix {SFT_IMATRIX} {SFT_GGUF_F16} {SFT_GGUF} Q4_K_M
print(f'SFT GGUF: {Path(SFT_GGUF).stat().st_size / 1e9:.2f} GB  →  {SFT_GGUF}')

# ── DPO model: F16 → imatrix → Q4_K_M ───────────────────────────────────
if not Path(DPO_GGUF).exists():
    if not Path(DPO_GGUF_F16).exists():
        print('Converting DPO → GGUF F16...')
        !python /content/llama.cpp/convert_hf_to_gguf.py \
            {DPO_MODEL_DIR} --outfile {DPO_GGUF_F16} --outtype f16
    if not Path(DPO_IMATRIX).exists():
        print('Generating DPO imatrix (shares calibration data)...')
        !{LLAMA_IMATRIX} -m {DPO_GGUF_F16} -f {CALIB_FILE} \
            -o {DPO_IMATRIX} --n-gpu-layers {GPU_LAYERS} --chunks 128
    print('Quantizing DPO → Q4_K_M with imatrix...')
    !{LLAMA_QUANT} --imatrix {DPO_IMATRIX} {DPO_GGUF_F16} {DPO_GGUF} Q4_K_M
print(f'DPO GGUF: {Path(DPO_GGUF).stat().st_size / 1e9:.2f} GB  →  {DPO_GGUF}')

# Optionally remove F16 intermediates to free Drive space (irreversible)
KEEP_F16 = True
if not KEEP_F16:
    for f16 in [SFT_GGUF_F16, DPO_GGUF_F16]:
        if Path(f16).exists():
            Path(f16).unlink()
            print(f'Removed F16 intermediate: {f16}')

In [ ]:
import json

with open(TEST_PATH) as f:
    test_cases = json.load(f)['cases']
print(f'Test examples: {len(test_cases)}')

In [ ]:
# ── Evaluation metric helpers
import re, json as _json, numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Format-compliance + matrix-correctness scoring (regex / matrix lookups)
from smartriz.eval.format_check import (
    score_format_compliance,
    score_principle_correctness,
    aggregate_scores,
)

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

# Canonical TRIZ principle name aliases (covers common phrasings)
_PRINCIPLE_NAMES = {
    1: ['segmentation', 'segmented'],
    2: ['taking out', 'extraction'],
    3: ['local quality'],
    4: ['asymmetry'],
    5: ['merging', 'consolidation'],
    6: ['universality'],
    7: ['nested doll', 'nesting'],
    8: ['anti-weight', 'counterweight'],
    9: ['preliminary anti-action', 'prior counteraction'],
    10: ['preliminary action', 'prior action'],
    11: ['beforehand cushioning', 'beforehand compensation'],
    12: ['equipotentiality'],
    13: ['the other way round', 'inversion'],
    14: ['spheroidality', 'curvature'],
    15: ['dynamics', 'dynamism'],
    16: ['partial or excessive actions'],
    17: ['another dimension', 'moving to another dimension'],
    18: ['mechanical vibration'],
    19: ['periodic action'],
    20: ['continuity of useful action'],
    21: ['skipping', 'rushing through'],
    22: ['blessing in disguise', 'convert harm to benefit'],
    23: ['feedback'],
    24: ['intermediary', 'mediator'],
    25: ['self-service'],
    26: ['copying'],
    27: ['cheap short-living', 'cheap disposable'],
    28: ['mechanics substitution', 'replacement of mechanical system'],
    29: ['pneumatics and hydraulics', 'pneumatics', 'hydraulics'],
    30: ['flexible shells', 'thin films'],
    31: ['porous materials'],
    32: ['color changes', 'optical properties'],
    33: ['homogeneity'],
    34: ['discarding and recovering', 'rejecting and regenerating'],
    35: ['parameter changes', 'transformation of properties'],
    36: ['phase transitions', 'phase transition'],
    37: ['thermal expansion'],
    38: ['strong oxidants', 'enriched atmosphere'],
    39: ['inert atmosphere', 'inert environment'],
    40: ['composite materials', 'composites'],
}

def extract_principle_numbers(text):
    found = set()
    t = text.lower()
    for m in re.finditer(r'#\s*(\d{1,2})', t):
        n = int(m.group(1))
        if 1 <= n <= 40: found.add(n)
    for m in re.finditer(r'\bprinciple\s+(\d{1,2})\b', t):
        n = int(m.group(1))
        if 1 <= n <= 40: found.add(n)
    for m in re.finditer(r'\b(\d{1,2})-[A-Za-z]', text):
        n = int(m.group(1))
        if 1 <= n <= 40: found.add(n)
    for num, aliases in _PRINCIPLE_NAMES.items():
        if any(alias in t for alias in aliases): found.add(num)
    return found

def principle_accuracy(generated, reference_principles):
    ref_nums = extract_principle_numbers(' '.join(reference_principles))
    gen_nums = extract_principle_numbers(generated)
    return len(ref_nums & gen_nums) >= min(2, len(ref_nums))

def contradiction_accuracy(generated, ref_cp):
    gen_lower = generated.lower()
    def _check_param(param_str):
        id_match = re.search(r'\(#(\d{1,2})\)', param_str)
        if id_match:
            n = id_match.group(1)
            return f'(#{n})' in generated or bool(re.search(rf'\b{n}\b', generated))
        content_words = [w for w in param_str.lower().split()
                         if len(w) >= 5 and not re.match(r'[#()\d]+', w)]
        if not content_words: return False
        matched = sum(1 for w in content_words if w in gen_lower)
        return matched >= max(1, int(len(content_words) * 0.6))
    return _check_param(ref_cp.get('improving_parameter', '')) and \
           _check_param(ref_cp.get('worsening_parameter', ''))

def solution_cosine(generated, reference_solution):
    embs = embed_model.encode([generated, reference_solution])
    return float(cosine_similarity([embs[0]], [embs[1]])[0][0])

# ── LLM-as-judge ──────────────────────────────────────────────────────────
_JUDGE_SYSTEM = (
    'You are an expert TRIZ methodology evaluator. '
    'Score the model response on four criteria, each 0-10. '
    'Respond ONLY with valid JSON, no extra text or markdown fences.'
)
_JUDGE_TEMPLATE = '''## Problem
{problem}

## Reference Inventive Principles
{reference_principles}

## Reference Contradiction
- Improving: {improving}
- Worsening: {worsening}

## Reference Solution (first 400 chars)
{reference_solution}

## Model Response
{generated}

Respond with JSON only:
{{"principle_quality": <0-10>, "contradiction_quality": <0-10>, "solution_quality": <0-10>, "reasoning_quality": <0-10>, "overall": <0-10>}}'''

def llm_judge_score(generated, case, client, model_name):
    cp = case.get('contradiction_pair', {})
    prompt = _JUDGE_TEMPLATE.format(
        problem=case['problem'][:800],
        reference_principles=', '.join(case.get('inventive_principles', [])),
        improving=cp.get('improving_parameter', ''),
        worsening=cp.get('worsening_parameter', ''),
        reference_solution=case.get('solution', '')[:400],
        generated=generated[:1200],
    )
    try:
        resp = client.chat.completions.create(
            model=model_name,
            messages=[{'role': 'system', 'content': _JUDGE_SYSTEM},
                      {'role': 'user',   'content': prompt}],
            temperature=0, max_tokens=200,
        )
        raw = resp.choices[0].message.content.strip()
        raw = re.sub(r'^```(?:json)?\s*', '', raw)
        raw = re.sub(r'\s*```$', '', raw)
        return _json.loads(raw)
    except Exception as e:
        print(f'LLM judge error: {e}')
        return None

def score_predictions(generations, cases, run_llm_judge=False, judge_client=None, judge_model=None):
    scores = []
    fmt_scores = []
    matrix_passes = []
    jaccards = []
    for gen, case in zip(generations, cases):
        cp = case['contradiction_pair']
        fmt = score_format_compliance(gen)
        pcorr = score_principle_correctness(
            gen,
            expected_improving=cp.get('improving_parameter', ''),
            expected_worsening=cp.get('worsening_parameter', ''),
            expected_principles=case.get('inventive_principles', []),
        )
        fmt_scores.append(fmt)
        matrix_passes.append(1.0 if pcorr['matrix_pass'] else 0.0)
        if pcorr['jaccard'] is not None:
            jaccards.append(pcorr['jaccard'])

        entry = {
            'principle_acc':     principle_accuracy(gen, case['inventive_principles']),
            'contradiction_acc': contradiction_accuracy(gen, cp),
            'cosine_sim':        solution_cosine(gen, case['solution']),
            'format_score':      fmt['score'],
            'matrix_pass':       pcorr['matrix_pass'],
            'generated_preview': gen[:300],
        }
        if run_llm_judge and judge_client is not None:
            entry['llm_judge'] = llm_judge_score(gen, case, judge_client, judge_model)
        scores.append(entry)

    fmt_agg = aggregate_scores(fmt_scores)
    result = {
        'principle_acc':     float(np.mean([s['principle_acc'] for s in scores])),
        'contradiction_acc': float(np.mean([s['contradiction_acc'] for s in scores])),
        'cosine_sim':        float(np.mean([s['cosine_sim'] for s in scores])),
        'format_compliance': fmt_agg,
        'principle_correctness': {
            'matrix_pass_rate': float(np.mean(matrix_passes)) if matrix_passes else 0.0,
            'jaccard_mean':     float(np.mean(jaccards)) if jaccards else None,
        },
        'chat_preamble_leak_rate': fmt_agg.get('chat_preamble_leak_rate', 0.0),
        'per_example':       scores,
    }
    if run_llm_judge:
        judge_scores = [s.get('llm_judge') for s in scores if s.get('llm_judge')]
        if judge_scores:
            for k in ['principle_quality', 'contradiction_quality', 'solution_quality',
                      'reasoning_quality', 'overall']:
                vals = [js[k] for js in judge_scores if js and k in js]
                result[f'llm_judge_{k}'] = float(np.mean(vals)) if vals else None
    return result

print('Metric helpers ready (format_check + matrix-correctness + LLM judge)')


In [ ]:
# ── Model runner utilities
import gc
import torch
from transformers import pipeline as hf_pipeline, AutoTokenizer
from openai import OpenAI
from tqdm.auto import tqdm

SYSTEM_PROMPT = (
    'You are SmarTRIZ, an expert engineering innovation assistant. '
    'Solve technical problems using TRIZ methodology. Identify the '
    'technical contradiction, select inventive principles from the '
    'Altshuller matrix, reason step by step, and propose a solution.'
)

def run_hf_model(model_path, cases, label=''):
    tok = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    gen = hf_pipeline('text-generation', model=model_path, tokenizer=tok,
                      device_map='auto', max_new_tokens=1024, do_sample=False)
    results = []
    try:
        for case in tqdm(cases, desc=f'eval {label or model_path.split("/")[-1]}'):
            prompt = tok.apply_chat_template(
                [{'role': 'system', 'content': SYSTEM_PROMPT},
                 {'role': 'user',   'content': case['problem']}],
                tokenize=False, add_generation_prompt=True
            )
            out = gen(prompt)[0]['generated_text']
            results.append(out[len(prompt):].strip())
    finally:
        del gen
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f'  Freed GPU memory after {label or "model"}')
    return results

def run_api_model(model_name, cases, label=''):
    client = OpenAI(api_key=DEEPINFRA_API_KEY,
                    base_url='https://api.deepinfra.com/v1/openai')
    results = []
    for case in tqdm(cases, desc=f'eval {label or model_name}'):
        try:
            resp = client.chat.completions.create(
                model=model_name,
                messages=[
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user',   'content': case['problem']},
                ],
                temperature=0, max_tokens=1024,
            )
            results.append(resp.choices[0].message.content)
        except Exception as e:
            results.append(f'API_ERROR: {e}')
    return results

print('Model runners ready')

In [ ]:
# ── Run evaluation — skips models already in results.json
from pathlib import Path
from openai import OpenAI

results_path = f'{EVAL_DIR}results.json'

if Path(results_path).exists():
    with open(results_path) as f:
        all_results = json.load(f)
    print(f'Loaded existing results for: {list(all_results.keys())}')
else:
    all_results = {}

# Initialize LLM judge client once (reused across all model evaluations)
judge_client = None
if DEEPINFRA_API_KEY and RUN_LLM_JUDGE:
    judge_client = OpenAI(api_key=DEEPINFRA_API_KEY,
                          base_url='https://api.deepinfra.com/v1/openai')
    print('LLM judge client initialized')

models_to_eval = [
    ('baseline', 'hf',  BASELINE_MODEL, 'baseline-qwen2.5'),
    ('sft',      'hf',  SFT_MODEL_DIR,  f'sft-{MODEL_SIZE}'),
    ('dpo',      'hf',  DPO_MODEL_DIR,  f'dpo-{MODEL_SIZE}'),
    ('teacher',  'api', TEACHER_MODEL,  'deepseek-r1-70b'),
]

for name, mode, path, label in models_to_eval:
    if name in all_results:
        print(f'Skipping {name} — already in results.json')
        continue
    print(f'\nEvaluating: {name} ({label})')
    generations = (run_hf_model(path, test_cases, label)
                   if mode == 'hf'
                   else run_api_model(path, test_cases, label))

    all_results[name] = score_predictions(
        generations, test_cases,
        run_llm_judge=(judge_client is not None),
        judge_client=judge_client,
        judge_model=JUDGE_MODEL,
    )

    with open(results_path, 'w') as f:
        json.dump(all_results, f, indent=2)

    r = all_results[name]
    print(f'  Principle Acc:     {r["principle_acc"]:.3f}')
    print(f'  Contradiction Acc: {r["contradiction_acc"]:.3f}')
    print(f'  Cosine Sim:        {r["cosine_sim"]:.3f}')
    if r.get('llm_judge_overall') is not None:
        print(f'  LLM Judge Overall: {r["llm_judge_overall"]:.2f}/10')

print(f'\nAll results saved to {results_path}')

In [ ]:
# ── Final comparison table
display_order = ['baseline', 'sft', 'dpo', 'teacher']
has_llm = any(all_results.get(n, {}).get('llm_judge_overall') is not None for n in display_order)

cols = [('Model', 12, '<'), ('Princ Acc', 9, '>'), ('Contra', 7, '>'),
        ('Cosine', 7, '>'), ('Format', 7, '>'), ('Matrix', 7, '>'),
        ('PreLeak', 8, '>')]
if has_llm:
    cols.append(('LLM Ovr', 8, '>'))

header = ' '.join(f'{c[0]:{c[2]}{c[1]}}' for c in cols)
print(header)
print('-' * len(header))

for name in display_order:
    if name not in all_results: continue
    r = all_results[name]
    fmt = r.get('format_compliance', {}).get('mean', float('nan'))
    mat = r.get('principle_correctness', {}).get('matrix_pass_rate', float('nan'))
    leak = r.get('chat_preamble_leak_rate', float('nan'))
    row = (
        f'{name:<12} '
        f'{r["principle_acc"]:>9.3f} '
        f'{r["contradiction_acc"]:>7.3f} '
        f'{r["cosine_sim"]:>7.3f} '
        f'{fmt:>7.3f} '
        f'{mat:>7.3f} '
        f'{leak:>8.3f}'
    )
    if has_llm:
        llm_val = r.get('llm_judge_overall')
        row += f' {llm_val:>8.2f}' if llm_val is not None else f' {"N/A":>8}'
    print(row)

print(f'\nFull results with per-example breakdowns → {results_path}')


In [ ]:
# ── Manual review: first 5 test cases × all evaluated models
from textwrap import shorten
review_n = 5
display_order = ['baseline', 'sft', 'dpo', 'teacher']

for i in range(review_n):
    case = test_cases[i]
    print('=' * 80)
    print(f'CASE {i+1}: {shorten(case["problem"], width=160, placeholder="…")}')
    print(f'EXPECTED principles: {case.get("inventive_principles", [])}')
    print(f'EXPECTED solution[:200]: {case.get("solution", "")[:200]}…')
    for name in display_order:
        if name not in all_results: continue
        r = all_results[name]
        per = r.get('per_example', [])
        if i >= len(per): continue
        prev = per[i].get('generated_preview', '')[:300]
        print(f'\n--- {name.upper()} (format={per[i].get("format_score", float("nan")):.2f} '
              f'matrix={per[i].get("matrix_pass", False)}) ---')
        print(prev.replace('\n', ' ⏎ '))
    print()
